# MetaTrader 5 - Interactive Tick Chart

This notebook demonstrates how to fetch tick data from **MetaTrader 5 (MT5)** and visualize an interactive tick price and spread chart using **Plotly**.

### Features:
1. Initialize connection to MT5 terminal.
2. Query tick range for a specific day (default: yesterday).
3. Preprocess tick data into a pandas DataFrame.
4. Calculate bid-ask spreads using universal formula (`mt5.symbol_info`).
5. Visualize tick prices (Bid/Ask step lines) and spread with an **interactive Plotly multi-panel chart** featuring synchronized zooming, panning, rich tooltips, and range slider.


In [50]:
import MetaTrader5 as mt5
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, UTC
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set pandas options for better display
pd.set_option('display.max_columns', 10)
pd.set_option('display.width', 1000)

In [51]:
# Initialize MetaTrader 5 connection
if not mt5.initialize():
    print("MetaTrader5 initialization failed, error code:", mt5.last_error())
    quit()
else:
    print("MetaTrader5 initialized successfully.")

    # Print terminal connection info
    info = mt5.terminal_info()
    print(f"Connected to: {info.company} - {info.name}")
    print(f"Version: {mt5.version()}")

MetaTrader5 initialized successfully.
Connected to: RoboForex Ltd - RoboForex MT5 Terminal
Version: (500, 6090, '31 Jul 2026')


## Define Parameters

By default, we fetch ticks for the `EURUSD` symbol for **yesterday** (the day prior to current execution).
You can customize the `SYMBOL` and date range below.

In [52]:
# --- Configuration ---
SYMBOL = "EURUSD"  # Symbol to fetch
today = datetime.now()
yesterday = today - timedelta(days=1)

# Start and end of yesterday
date_from = datetime(yesterday.year, yesterday.month, yesterday.day, 0, 0, 0, tzinfo=UTC)
date_to = datetime(yesterday.year, yesterday.month, yesterday.day, 23, 59, 59, tzinfo=UTC)

print(f"Symbol: {SYMBOL}")
print(f"Date From: {date_from}")
print(f"Date To:   {date_to}")

Symbol: EURUSD
Date From: 2026-08-12 00:00:00+00:00
Date To:   2026-08-12 23:59:59+00:00


## Fetch Tick Data

We use `mt5.copy_ticks_range` to request tick data. The flag `mt5.COPY_TICKS_ALL` retrieves all tick types (bid, ask, etc.).

In [53]:
print(f"Requesting tick data for {SYMBOL}...")
ticks = mt5.copy_ticks_range(SYMBOL, date_from, date_to, mt5.COPY_TICKS_ALL)

if ticks is None or len(ticks) == 0:
    print(f"No ticks retrieved. Check if {SYMBOL} is available in Market Watch or if the market was open during the selected range.")
    print("Error code:", mt5.last_error())
else:
    print(f"Successfully retrieved {len(ticks):,} ticks.")

Requesting tick data for EURUSD...
Successfully retrieved 53,700 ticks.


## Preprocess Data

Let's convert the fetched ticks array into a pandas DataFrame, format the timestamps, and compute the raw spread (`ask - bid`).

To make the calculation universal, we fetch the symbol's information using `mt5.symbol_info` to get its `point` size and `digits` count. A standard **pip** is typically defined as 10 points for currency pairs with 3 or 5 decimal digits, and as 1 point for other symbols (such as gold, crypto, or indices).

In [54]:
if ticks is not None and len(ticks) > 0:
    # Create DataFrame
    df = pd.DataFrame(ticks)

    # Convert millisecond timestamp to pandas datetime
    df['time'] = pd.to_datetime(df['time_msc'], unit='ms')

    # Set the time column as the index for easier analysis
    df.set_index('time', inplace=True)

    # Fetch instrument info for universal pip calculation
    info = mt5.symbol_info(SYMBOL)
    if info is not None:
        point = info.point
        # A standard pip is 10 points for 3/5 digit forex pairs, and 1 point for others
        if info.digits in [3, 5]:
            pip_size = 10 * point
        else:
            pip_size = point
        print(f"Universal Pip Calculation: 1 Pip = {pip_size} (Point: {point}, Digits: {info.digits})")
    else:
        pip_size = 0.0001
        print(f"Symbol info not found for {SYMBOL}. Using fallback 1 Pip = {pip_size}")

    # Calculate bid-ask spread
    df['spread_raw'] = df['ask'] - df['bid']
    df['spread_pips'] = df['spread_raw'] / pip_size

    # Display the first few rows
    print("\nPreprocessed Tick DataFrame:")
    display(df.head())
else:
    print("No data to preprocess.")

Universal Pip Calculation: 1 Pip = 0.0001 (Point: 1e-05, Digits: 5)

Preprocessed Tick DataFrame:


,bid,ask,last,volume,time_msc,flags,volume_real,spread_raw,spread_pips
time,,,,,,,,,
2026-08-12 00:05:00.428,1.15392,1.15436,0.0,0,1786493100428,134,0.0,0.00044,4.4
2026-08-12 00:05:10.540,1.15393,1.15437,0.0,0,1786493110540,134,0.0,0.00044,4.4
2026-08-12 00:05:41.037,1.15399,1.15431,0.0,0,1786493141037,134,0.0,0.00032,3.2
2026-08-12 00:05:41.133,1.15401,1.15429,0.0,0,1786493141133,134,0.0,0.00028,2.8
2026-08-12 00:05:41.229,1.15402,1.15428,0.0,0,1786493141229,134,0.0,0.00026,2.6


## Summary Statistics

Let's look at the basic statistics of the ticks, such as average, maximum, and minimum spreads.

In [55]:
if ticks is not None and len(ticks) > 0:
    print("--- Summary Statistics ---")
    print(f"Total Ticks: {len(df):,}")
    print(f"Min Bid: {df['bid'].min():.5f}")
    print(f"Max Ask: {df['ask'].max():.5f}")
    print(f"Average Spread (pips): {df['spread_pips'].mean():.2f}")
    print(f"Max Spread (pips): {df['spread_pips'].max():.2f}")
    print(f"Min Spread (pips): {df['spread_pips'].min():.2f}")
else:
    print("No data available for statistics.")

--- Summary Statistics ---
Total Ticks: 53,700
Min Bid: 1.15196
Max Ask: 1.15629
Average Spread (pips): 0.14
Max Spread (pips): 4.40
Min Spread (pips): 0.00


## Visualizing the Interactive Tick Chart

We construct an **interactive multi-panel display** using **Plotly**:
1. **Price Action (Bid & Ask)** rendered using step lines (`shape='hv'`) to accurately reflect discrete price changes between ticks when zooming in.
2. **Bid-Ask Spread** (in pips) over time.

Both subplots share a **synchronized X-axis**. Zooming or panning on either panel updates both panels dynamically.

In [56]:
if ticks is not None and len(ticks) > 0:
    # Create 2-row subplot figure with synchronized x-axes
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        row_heights=[0.68, 0.32],
        subplot_titles=(f"{SYMBOL} Tick Price Action (Bid & Ask)", "Bid-Ask Spread (Pips)")
    )

    # Custom hover data array for Bid trace: [bid, ask, spread_pips]
    custom_price = np.stack((df['bid'], df['ask'], df['spread_pips']), axis=-1)

    # 1. Bid Price (Step Line) - Tooltip shows Time, Bid Price, Spread
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df['bid'],
            mode='lines',
            name='Bid',
            line=dict(color='#1f77b4', shape='hv', width=1.5),
            customdata=custom_price,
            hovertemplate=(
                "<b>Time:</b> %{x|%Y-%m-%d %H:%M:%S.%3f}<br>" +
                "<b>Bid:</b> %{y:.5f}<br>" +
                "<b>Spread:</b> %{customdata[2]:.2f} pips<extra>Bid</extra>"
            )
        ),
        row=1, col=1
    )

    # 2. Ask Price (Step Line) - Tooltip shows Ask Price
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df['ask'],
            mode='lines',
            name='Ask',
            line=dict(color='#ff7f0e', shape='hv', width=1.5),
            hovertemplate=(
                "<b>Ask:</b> %{y:.5f}<extra>Ask</extra>"
            )
        ),
        row=1, col=1
    )

    # 3. Spread (Pips) - Tooltip shows Time and Spread Value
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df['spread_pips'],
            mode='lines',
            name='Spread (pips)',
            line=dict(color='#2ca02c', shape='hv', width=1.2),
            hovertemplate=(
                "<b>Time:</b> %{x|%Y-%m-%d %H:%M:%S.%3f}<br>" +
                "<b>Spread:</b> %{y:.2f} pips<extra>Spread</extra>"
            )
        ),
        row=2, col=1
    )

    # Update Layout & Interactive Controls
    date_str = df.index[0].strftime('%Y-%m-%d') if len(df) > 0 else ""
    fig.update_layout(
        title=dict(
            text=f"MetaTrader 5 - {SYMBOL} Interactive Tick Chart ({date_str})",
            x=0.5,
            xanchor='center'
        ),
        height=750,
        hovermode='x unified',
        template='plotly_white',
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=1.02,
            xanchor='right',
            x=1
        ),
        margin=dict(l=60, r=40, t=100, b=60)
    )

    # Configure X-axis Range Slider on bottom subplot
    fig.update_xaxes(
        row=2, col=1,
        rangeslider=dict(visible=True, thickness=0.08),
        type='date'
    )

    # Y-axes labels
    fig.update_yaxes(title_text="Price", row=1, col=1)
    fig.update_yaxes(title_text="Spread (pips)", row=2, col=1)

    fig.show()
else:
    print("No tick data available to visualize.")